In [1]:
import nbimport
import os, sys, json

In [2]:
from rag import collection

Note: you may need to restart the kernel to use updated packages.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
%pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [4]:
ppp_json = """{
  "problem_name": "Production_Planning",
  "sense": "min",
  "variables": [
    {
      "name": "x",
      "index": ["A_North", "A_South", "B_North", "B_South", "C_North", "C_South"],
      "type": "continuous",
      "lb": 0,
      "ub": null
    }
  ],
  "objective": {
    "terms": [
      {"var": "x_A_North", "coeff": 5},
      {"var": "x_A_South", "coeff": 7},
      {"var": "x_B_North", "coeff": 9},
      {"var": "x_B_South", "coeff": 6},
      {"var": "x_C_North", "coeff": 8},
      {"var": "x_C_South", "coeff": 10}
    ],
    "constant": 0
  },
  "constraints": [
    {
      "name": "Demand_A",
      "terms": [
        {"var": "x_A_North", "coeff": 1},
        {"var": "x_A_South", "coeff": 1}
      ],
      "rhs": 400,
      "sense": ">="
    },
    {
      "name": "Demand_B",
      "terms": [
        {"var": "x_B_North", "coeff": 1},
        {"var": "x_B_South", "coeff": 1}
      ],
      "rhs": 300,
      "sense": ">="
    },
    {
      "name": "Demand_C",
      "terms": [
        {"var": "x_C_North", "coeff": 1},
        {"var": "x_C_South", "coeff": 1}
      ],
      "rhs": 500,
      "sense": ">="
    },
    {
      "name": "Capacity_North",
      "terms": [
        {"var": "x_A_North", "coeff": 1},
        {"var": "x_B_North", "coeff": 1},
        {"var": "x_C_North", "coeff": 1}
      ],
      "rhs": 700,
      "sense": "<="
    },
    {
      "name": "Capacity_South",
      "terms": [
        {"var": "x_A_South", "coeff": 1},
        {"var": "x_B_South", "coeff": 1},
        {"var": "x_C_South", "coeff": 1}
      ],
      "rhs": 800,
      "sense": "<="
    }
  ]
}"""

ppp_latex = r"""\min \quad
5 x_{A,\text{North}} + 7 x_{A,\text{South}}
+ 9 x_{B,\text{North}} + 6 x_{B,\text{South}}
+ 8 x_{C,\text{North}} + 10 x_{C,\text{South}}

\text{s.t.}

x_{A,\text{North}} + x_{A,\text{South}} \geq 400

x_{B,\text{North}} + x_{B,\text{South}} \geq 300

x_{C,\text{North}} + x_{C,\text{South}} \geq 500

x_{A,\text{North}} + x_{B,\text{North}} + x_{C,\text{North}} \leq 700

x_{A,\text{South}} + x_{B,\text{South}} + x_{C,\text{South}} \leq 800

x_{p,l} \geq 0 \qquad \forall\, p \in \{A,B,C\},\ l \in \{\text{North}, \text{South}\}"""


vrp_json = """{
  "problem_name": "Small_CVRP_5nodes_2vehicles",
  "sense": "min",
  "variables": [
    {
      "name": "x",
      "index": [
        "0_1_1", "0_2_1", "0_3_1", "0_4_1",
        "1_0_1", "1_2_1", "1_3_1", "1_4_1",
        "2_0_1", "2_1_1", "2_3_1", "2_4_1",
        "3_0_1", "3_1_1", "3_2_1", "3_4_1",
        "4_0_1", "4_1_1", "4_2_1", "4_3_1",
        "0_1_2", "0_2_2", "0_3_2", "0_4_2",
        "1_0_2", "1_2_2", "1_3_2", "1_4_2",
        "2_0_2", "2_1_2", "2_3_2", "2_4_2",
        "3_0_2", "3_1_2", "3_2_2", "3_4_2",
        "4_0_2", "4_1_2", "4_2_2", "4_3_2"
      ],
      "type": "binary",
      "lb": 0,
      "ub": 1
    }
  ],
  "objective": {
    "terms": [
      {"var": "x_0_1_1", "coeff": 12}, {"var": "x_0_2_1", "coeff": 15},
      {"var": "x_0_3_1", "coeff": 18}, {"var": "x_0_4_1", "coeff": 10},
      {"var": "x_1_0_1", "coeff": 12}, {"var": "x_1_2_1", "coeff": 8},
      {"var": "x_1_3_1", "coeff": 14}, {"var": "x_1_4_1", "coeff": 9},
      {"var": "x_2_0_1", "coeff": 15}, {"var": "x_2_1_1", "coeff": 8},
      {"var": "x_2_3_1", "coeff": 11}, {"var": "x_2_4_1", "coeff": 13},
      {"var": "x_3_0_1", "coeff": 18}, {"var": "x_3_1_1", "coeff": 14},
      {"var": "x_3_2_1", "coeff": 11}, {"var": "x_3_4_1", "coeff": 7},
      {"var": "x_4_0_1", "coeff": 10}, {"var": "x_4_1_1", "coeff": 9},
      {"var": "x_4_2_1", "coeff": 13}, {"var": "x_4_3_1", "coeff": 7},
      {"var": "x_0_1_2", "coeff": 12}, {"var": "x_0_2_2", "coeff": 15},
      {"var": "x_0_3_2", "coeff": 18}, {"var": "x_0_4_2", "coeff": 10},
      {"var": "x_1_0_2", "coeff": 12}, {"var": "x_1_2_2", "coeff": 8},
      {"var": "x_1_3_2", "coeff": 14}, {"var": "x_1_4_2", "coeff": 9},
      {"var": "x_2_0_2", "coeff": 15}, {"var": "x_2_1_2", "coeff": 8},
      {"var": "x_2_3_2", "coeff": 11}, {"var": "x_2_4_2", "coeff": 13},
      {"var": "x_3_0_2", "coeff": 18}, {"var": "x_3_1_2", "coeff": 14},
      {"var": "x_3_2_2", "coeff": 11}, {"var": "x_3_4_2", "coeff": 7},
      {"var": "x_4_0_2", "coeff": 10}, {"var": "x_4_1_2", "coeff": 9},
      {"var": "x_4_2_2", "coeff": 13}, {"var": "x_4_3_2", "coeff": 7}
    ],
    "constant": 0
  },
  "constraints": [
    {"name": "Leave_depot_v1", "terms": [{"var": "x_0_1_1", "coeff": 1}, {"var": "x_0_2_1", "coeff": 1}, {"var": "x_0_3_1", "coeff": 1}, {"var": "x_0_4_1", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Leave_depot_v2", "terms": [{"var": "x_0_1_2", "coeff": 1}, {"var": "x_0_2_2", "coeff": 1}, {"var": "x_0_3_2", "coeff": 1}, {"var": "x_0_4_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Return_depot_v1", "terms": [{"var": "x_1_0_1", "coeff": 1}, {"var": "x_2_0_1", "coeff": 1}, {"var": "x_3_0_1", "coeff": 1}, {"var": "x_4_0_1", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Return_depot_v2", "terms": [{"var": "x_1_0_2", "coeff": 1}, {"var": "x_2_0_2", "coeff": 1}, {"var": "x_3_0_2", "coeff": 1}, {"var": "x_4_0_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Visit_1", "terms": [{"var": "x_0_1_1", "coeff": 1}, {"var": "x_2_1_1", "coeff": 1}, {"var": "x_3_1_1", "coeff": 1}, {"var": "x_4_1_1", "coeff": 1}, {"var": "x_0_1_2", "coeff": 1}, {"var": "x_2_1_2", "coeff": 1}, {"var": "x_3_1_2", "coeff": 1}, {"var": "x_4_1_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Visit_2", "terms": [{"var": "x_0_2_1", "coeff": 1}, {"var": "x_1_2_1", "coeff": 1}, {"var": "x_3_2_1", "coeff": 1}, {"var": "x_4_2_1", "coeff": 1}, {"var": "x_0_2_2", "coeff": 1}, {"var": "x_1_2_2", "coeff": 1}, {"var": "x_3_2_2", "coeff": 1}, {"var": "x_4_2_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Visit_3", "terms": [{"var": "x_0_3_1", "coeff": 1}, {"var": "x_1_3_1", "coeff": 1}, {"var": "x_2_3_1", "coeff": 1}, {"var": "x_4_3_1", "coeff": 1}, {"var": "x_0_3_2", "coeff": 1}, {"var": "x_1_3_2", "coeff": 1}, {"var": "x_2_3_2", "coeff": 1}, {"var": "x_4_3_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Visit_4", "terms": [{"var": "x_0_4_1", "coeff": 1}, {"var": "x_1_4_1", "coeff": 1}, {"var": "x_2_4_1", "coeff": 1}, {"var": "x_3_4_1", "coeff": 1}, {"var": "x_0_4_2", "coeff": 1}, {"var": "x_1_4_2", "coeff": 1}, {"var": "x_2_4_2", "coeff": 1}, {"var": "x_3_4_2", "coeff": 1}], "rhs": 1, "sense": "=="},
    {"name": "Flow_1", "terms": [{"var": "x_0_1_1", "coeff": 1}, {"var": "x_2_1_1", "coeff": 1}, {"var": "x_3_1_1", "coeff": 1}, {"var": "x_4_1_1", "coeff": 1}, {"var": "x_1_0_1", "coeff": -1}, {"var": "x_1_2_1", "coeff": -1}, {"var": "x_1_3_1", "coeff": -1}, {"var": "x_1_4_1", "coeff": -1}], "rhs": 0, "sense": "=="},
    {"name": "Flow_2", "terms": [{"var": "x_0_2_1", "coeff": 1}, {"var": "x_1_2_1", "coeff": 1}, {"var": "x_3_2_1", "coeff": 1}, {"var": "x_4_2_1", "coeff": 1}, {"var": "x_2_0_1", "coeff": -1}, {"var": "x_2_1_1", "coeff": -1}, {"var": "x_2_3_1", "coeff": -1}, {"var": "x_2_4_1", "coeff": -1}], "rhs": 0, "sense": "=="},
    {"name": "Flow_3", "terms": [{"var": "x_0_3_1", "coeff": 1}, {"var": "x_1_3_1", "coeff": 1}, {"var": "x_2_3_1", "coeff": 1}, {"var": "x_4_3_1", "coeff": 1}, {"var": "x_3_0_1", "coeff": -1}, {"var": "x_3_1_1", "coeff": -1}, {"var": "x_3_2_1", "coeff": -1}, {"var": "x_3_4_1", "coeff": -1}], "rhs": 0, "sense": "=="},
    {"name": "Flow_4", "terms": [{"var": "x_0_4_1", "coeff": 1}, {"var": "x_1_4_1", "coeff": 1}, {"var": "x_2_4_1", "coeff": 1}, {"var": "x_3_4_1", "coeff": 1}, {"var": "x_4_0_1", "coeff": -1}, {"var": "x_4_1_1", "coeff": -1}, {"var": "x_4_2_1", "coeff": -1}, {"var": "x_4_3_1", "coeff": -1}], "rhs": 0, "sense": "=="},
    {
      "name": "Capacity_veh1",
      "terms": [
        {"var": "x_0_1_1", "coeff": 2}, {"var": "x_0_2_1", "coeff": 3}, {"var": "x_0_3_1", "coeff": 4}, {"var": "x_0_4_1", "coeff": 1},
        {"var": "x_1_2_1", "coeff": 2}, {"var": "x_1_3_1", "coeff": 2}, {"var": "x_1_4_1", "coeff": 2},
        {"var": "x_2_3_1", "coeff": 3}, {"var": "x_2_4_1", "coeff": 3},
        {"var": "x_3_4_1", "coeff": 4}
      ],
      "rhs": 10,
      "sense": "<="
    },
    {
      "name": "Capacity_veh2",
      "terms": [
        {"var": "x_0_1_2", "coeff": 2}, {"var": "x_0_2_2", "coeff": 3}, {"var": "x_0_3_2", "coeff": 4}, {"var": "x_0_4_2", "coeff": 1},
        {"var": "x_1_2_2", "coeff": 2}, {"var": "x_1_3_2", "coeff": 2}, {"var": "x_1_4_2", "coeff": 2},
        {"var": "x_2_3_2", "coeff": 3}, {"var": "x_2_4_2", "coeff": 3},
        {"var": "x_3_4_2", "coeff": 4}
      ],
      "rhs": 10,
      "sense": "<="
    }
  ]
}"""

vrp_latex = r"""\min \sum_{i \in V} \sum_{j \in V \setminus \{i\}} \sum_{k \in K} c_{ij}\, x_{ijk}

\text{s.t.}

\sum_{k \in K} \sum_{j \in V \setminus \{i\}} x_{ijk} = 1 \qquad \forall i \in N

\sum_{j \in V \setminus \{0\}} x_{0jk} = 1 \qquad \forall k \in K

\sum_{i \in V \setminus \{0\}} x_{i0k} = 1 \qquad \forall k \in K

\sum_{j \in V \setminus \{i\}} x_{ijk} - \sum_{j \in V \setminus \{i\}} x_{jik} = 0 \qquad \forall i \in N,\ \forall k \in K

\sum_{i \in N} q_i \sum_{j \in V \setminus \{i\}} x_{ijk} \leq Q \qquad \forall k \in K

x_{ijk} \in \{0,1\} \qquad \forall i,j \in V,\ i \neq j,\ \forall k \in K

\begin{alignedat}{2}
& V = \{0,1,2,3,4\} && \quad (0 = \text{depot}) \\
& N = \{1,2,3,4\} && \\
& K = \{1,2\} && \quad (\text{vehicles}) \\
& Q &= 10 && \\
& q &= (2,3,4,1) && \text{(demands of nodes 1,2,3,4)}
\end{alignedat}"""


In [5]:
ppp_latex

'\\min \\quad\n5 x_{A,\\text{North}} + 7 x_{A,\\text{South}}\n+ 9 x_{B,\\text{North}} + 6 x_{B,\\text{South}}\n+ 8 x_{C,\\text{North}} + 10 x_{C,\\text{South}}\n\n\\text{s.t.}\n\nx_{A,\\text{North}} + x_{A,\\text{South}} \\geq 400\n\nx_{B,\\text{North}} + x_{B,\\text{South}} \\geq 300\n\nx_{C,\\text{North}} + x_{C,\\text{South}} \\geq 500\n\nx_{A,\\text{North}} + x_{B,\\text{North}} + x_{C,\\text{North}} \\leq 700\n\nx_{A,\\text{South}} + x_{B,\\text{South}} + x_{C,\\text{South}} \\leq 800\n\nx_{p,l} \\geq 0 \\qquad \\forall\\, p \\in \\{A,B,C\\},\\ l \\in \\{\\text{North}, \\text{South}\\}'

In [6]:
x = json.dumps(json.loads(ppp_json)) 
# make it easier to include in json raw strings
x = x.replace('"', '\\"')
print(x)

{\"problem_name\": \"Production_Planning\", \"sense\": \"min\", \"variables\": [{\"name\": \"x\", \"index\": [\"A_North\", \"A_South\", \"B_North\", \"B_South\", \"C_North\", \"C_South\"], \"type\": \"continuous\", \"lb\": 0, \"ub\": null}], \"objective\": {\"terms\": [{\"var\": \"x_A_North\", \"coeff\": 5}, {\"var\": \"x_A_South\", \"coeff\": 7}, {\"var\": \"x_B_North\", \"coeff\": 9}, {\"var\": \"x_B_South\", \"coeff\": 6}, {\"var\": \"x_C_North\", \"coeff\": 8}, {\"var\": \"x_C_South\", \"coeff\": 10}], \"constant\": 0}, \"constraints\": [{\"name\": \"Demand_A\", \"terms\": [{\"var\": \"x_A_North\", \"coeff\": 1}, {\"var\": \"x_A_South\", \"coeff\": 1}], \"rhs\": 400, \"sense\": \">=\"}, {\"name\": \"Demand_B\", \"terms\": [{\"var\": \"x_B_North\", \"coeff\": 1}, {\"var\": \"x_B_South\", \"coeff\": 1}], \"rhs\": 300, \"sense\": \">=\"}, {\"name\": \"Demand_C\", \"terms\": [{\"var\": \"x_C_North\", \"coeff\": 1}, {\"var\": \"x_C_South\", \"coeff\": 1}], \"rhs\": 500, \"sense\": \">=\

In [7]:
"""Small Capacitated Vehicle Routing Problem (CVRP) instance with:
- 1 depot (node 0)
- 4 customers (nodes 1-4)
- 2 identical vehicles
- Vehicle capacity: 10 units each
- Customer demands: node 1=2, node 2=3, node 3=4, node 4=1
- Symmetric distance matrix (travel costs)
- Objective: minimize total travel distance
- Constraints: each customer visited exactly once, each vehicle leaves and returns to depot exactly once, flow conservation per vehicle, total demand per vehicle ≤ capacity
- Formulation: basic three-index arc-flow model (no subtour elimination constraints included)
- Total nodes: 5
- Used in demonstration of MILP modeling for routing problems"""

'Small Capacitated Vehicle Routing Problem (CVRP) instance with:\n- 1 depot (node 0)\n- 4 customers (nodes 1-4)\n- 2 identical vehicles\n- Vehicle capacity: 10 units each\n- Customer demands: node 1=2, node 2=3, node 3=4, node 4=1\n- Symmetric distance matrix (travel costs)\n- Objective: minimize total travel distance\n- Constraints: each customer visited exactly once, each vehicle leaves and returns to depot exactly once, flow conservation per vehicle, total demand per vehicle ≤ capacity\n- Formulation: basic three-index arc-flow model (no subtour elimination constraints included)\n- Total nodes: 5\n- Used in demonstration of MILP modeling for routing problems'

In [8]:
from pulp import *
from typing import Dict, Any, List


def solve_milp_from_structured_input(data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Takes structured JSON-like dict and builds + solves MILP with PuLP
    Returns result dictionary
    """
    prob = LpProblem(
        data.get("problem_name", "Anonymous_Model"),
        (
            LpMinimize
            if data.get("sense", "min").lower() in ["min", "minimize"]
            else LpMaximize
        ),
    )

    # ── Create variables ───────────────────────────────────────────────
    var_dict = {}  # name → LpVariable

    for vdef in data.get("variables", []):
        vname = vdef["name"]
        indices = vdef.get("index", [])
        cat_map = {"continuous": "Continuous", "integer": "Integer", "binary": "Binary"}
        cat = cat_map.get(vdef.get("type", "continuous").lower(), "Continuous")

        if indices:
            # indexed variable
            keys = [
                tuple(idx.split("_")) if "_" in idx else idx for idx in indices
            ]  # simple split heuristic
            var_dict[vname] = LpVariable.dicts(
                name=vname,
                indices=indices,  # can be list of str or tuples
                lowBound=vdef.get("lb", None),
                upBound=vdef.get("ub", None),
                cat=cat,
            )
        else:
            # single variable
            var_dict[vname] = LpVariable(
                name=vname,
                lowBound=vdef.get("lb", None),
                upBound=vdef.get("ub", None),
                cat=cat,
            )

    # ── Objective ──────────────────────────────────────────────────────
    obj = LpAffineExpression()
    for term in data.get("objective", {}).get("terms", []):
        var_name = term["var"]
        # handle indexed names like x_A_North → x["A_North"] or x[("A","North")]
        if "_" in var_name and var_name.startswith(vdef["name"] + "_"):
            key = var_name.split("_", 1)[1]
            obj += term["coeff"] * var_dict[vdef["name"]][key]
        else:
            obj += term["coeff"] * var_dict[var_name]
    obj += data.get("objective", {}).get("constant", 0)
    prob += obj, "Objective"

    # ── Constraints ────────────────────────────────────────────────────
    for c in data.get("constraints", []):
        expr = LpAffineExpression()
        for term in c.get("terms", []):
            var_name = term["var"]
            if "_" in var_name and var_name.startswith(vdef["name"] + "_"):
                key = var_name.split("_", 1)[1]
                expr += term["coeff"] * var_dict[vdef["name"]][key]
            else:
                expr += term["coeff"] * var_dict[var_name]
        sense_map = {">=": ">=", "<=": "<=", "==": "=="}
        prob += (
            expr == c["rhs"]
            if c["sense"] == "=="
            else expr >= c["rhs"] if c["sense"] == ">=" else expr <= c["rhs"]
        ), c.get("name", "Constraint")

    # ── Solve ──────────────────────────────────────────────────────────
    status = prob.solve(PULP_CBC_CMD(msg=0))

    result = {
        "status": LpStatus[status],
        "objective": value(prob.objective) if status == LpStatusOptimal else None,
        "variables": {
            v.name: v.varValue for v in prob.variables() if v.varValue is not None
        },
        "solver_message": LpStatus[status],
    }

    return result


solve_milp_from_structured_input(json.loads(vrp_json))

{'status': 'Optimal',
 'objective': 57.0,
 'variables': {'x_0_1_1': 0.0,
  'x_0_1_2': 1.0,
  'x_0_2_1': 0.0,
  'x_0_2_2': 0.0,
  'x_0_3_1': 0.0,
  'x_0_3_2': 0.0,
  'x_0_4_1': 1.0,
  'x_0_4_2': 0.0,
  'x_1_0_1': 0.0,
  'x_1_0_2': 0.0,
  'x_1_2_1': 0.0,
  'x_1_2_2': 1.0,
  'x_1_3_1': 0.0,
  'x_1_3_2': 0.0,
  'x_1_4_1': 0.0,
  'x_1_4_2': 0.0,
  'x_2_0_1': 0.0,
  'x_2_0_2': 0.0,
  'x_2_1_1': 0.0,
  'x_2_1_2': 0.0,
  'x_2_3_1': 0.0,
  'x_2_3_2': 0.0,
  'x_2_4_1': 0.0,
  'x_2_4_2': 0.0,
  'x_3_0_1': 0.0,
  'x_3_0_2': 0.0,
  'x_3_1_1': 0.0,
  'x_3_1_2': 0.0,
  'x_3_2_1': 0.0,
  'x_3_2_2': 0.0,
  'x_3_4_1': 0.0,
  'x_3_4_2': 0.0,
  'x_4_0_1': 1.0,
  'x_4_0_2': 1.0,
  'x_4_1_1': 0.0,
  'x_4_1_2': 0.0,
  'x_4_2_1': 0.0,
  'x_4_2_2': 0.0,
  'x_4_3_1': 0.0,
  'x_4_3_2': 1.0},
 'solver_message': 'Optimal'}

In [9]:
print(vrp_latex)

\min \sum_{i \in V} \sum_{j \in V \setminus \{i\}} \sum_{k \in K} c_{ij}\, x_{ijk}

\text{s.t.}

\sum_{k \in K} \sum_{j \in V \setminus \{i\}} x_{ijk} = 1 \qquad \forall i \in N

\sum_{j \in V \setminus \{0\}} x_{0jk} = 1 \qquad \forall k \in K

\sum_{i \in V \setminus \{0\}} x_{i0k} = 1 \qquad \forall k \in K

\sum_{j \in V \setminus \{i\}} x_{ijk} - \sum_{j \in V \setminus \{i\}} x_{jik} = 0 \qquad \forall i \in N,\ \forall k \in K

\sum_{i \in N} q_i \sum_{j \in V \setminus \{i\}} x_{ijk} \leq Q \qquad \forall k \in K

x_{ijk} \in \{0,1\} \qquad \forall i,j \in V,\ i \neq j,\ \forall k \in K

\begin{alignedat}{2}
& V = \{0,1,2,3,4\} && \quad (0 = \text{depot}) \\
& N = \{1,2,3,4\} && \\
& K = \{1,2\} && \quad (\text{vehicles}) \\
& Q &= 10 && \\
& q &= (2,3,4,1) && \text{(demands of nodes 1,2,3,4)}
\end{alignedat}
